# 02 — Grating Spectrum with Broad-Line Detection

This notebook demonstrates fitting G395M grating spectra with automatic
BIC-based broad Balmer component detection.

The G395M grating has R ≈ 1000 (constant), resolving [NII]–Hα–[NII] and
[SII] doublets that are blended in the prism.  The broad component fitting
uses `fit_with_broad(mode="auto")` which compares narrow-only vs broad
models via BIC and selects the best one in a single call.

In [1]:
import jwspecfit
import matplotlib.pyplot as plt
import numpy as np

## Load the G395M spectrum

In [ ]:
spec = jwspecfit.read_fits(
    "../../data/excels-uds04-v4_g395m-f290lp_3543_63107.spec.fits",
    z=8.271,
)
print(f"Grating: {spec.grating}, {spec.n_pix} pixels")
print(f"Wave: {spec.wave_um.min():.2f} – {spec.wave_um.max():.2f} µm")

## Fit with broad component detection

`fit_with_broad(mode="auto")` runs all model variants in one call:
- **narrow**: narrow lines only
- **broad1**: narrow + intermediate broad Balmer (σ_v ≈ 3× narrow)
- **broad2**: narrow + very broad Balmer (σ_v ≈ 7× narrow)
- **both**: narrow + both broad components

The best model is selected by BIC with ΔBIC ≥ 6 threshold.
NII kinematics are tied to OIII to prevent the broad Hα from
absorbing NII flux.

In [ ]:
result = jwspecfit.fit_with_broad(spec, z=8.271, mode="auto")

print(f"Selected model: {result.selected_model}")
print(f"BIC narrow:     {result.bic_narrow:.1f}")
print(f"BIC broad1:     {result.bic_broad1:.1f}")
print(f"BIC broad2:     {result.bic_broad2:.1f}")
print(f"BIC both:       {result.bic_both:.1f}")
print()
print(f"χ²/dof = {result.best_fit.chi2:.2f}")
for name, lr in result.best_fit.lines.items():
    if lr.snr > 1:
        print(f"  {name:<18s} flux={lr.flux:.2e} ± {lr.flux_err:.2e}  SNR={lr.snr:.1f}")

In [ ]:
fig = jwspecfit.plot_fit(result.best_fit)
plt.show()

## Second G395M spectrum

In [2]:
spec2 = jwspecfit.read_fits(
    "../../data/stark-rxcj2248-v4_g395m-f290lp_2478_3.spec.fits",
    z=6.1052,
)
result2 = jwspecfit.fit_with_broad(spec2, z=6.1052, mode="auto")

print(f"Selected model: {result2.selected_model}")
print(f"χ²/dof = {result2.best_fit.chi2:.2f}")
for name, lr in result2.best_fit.lines.items():
    if lr.snr > 1:
        print(f"  {name:<18s} flux={lr.flux:.2e} ± {lr.flux_err:.2e}  SNR={lr.snr:.1f}")

fig = jwspecfit.plot_fit(result2.best_fit)
plt.show()

KeyboardInterrupt: 

## Interactive plot

Use the plotly interactive plot to zoom into the Hα+[NII] complex
and inspect narrow vs broad components.

In [ ]:
fig_interactive = jwspecfit.plot_fit_interactive(result.best_fit)
fig_interactive.show()

## Save results and export line table

In [ ]:
# Save the fit result for replotting later
jwspecfit.save_result(result.best_fit, "g395m_fit_result.npz")

# Export line measurements
jwspecfit.export_lines_txt(result.best_fit, "g395m_lines.txt")

with open("g395m_lines.txt") as f:
    print(f.read())